# LangChain으로 배우는 LLM 에이전트 기초

이 노트북은 LLM(대형 언어 모델)을 활용한 에이전트 개발의 핵심 개념을 단계별로 학습합니다.

| 섹션 | 내용 |
|------|------|
| 1 | LLM 호출 — 모델에 질문하고 답 받기 |
| 2 | 도구 호출 — LLM이 함수를 선택하게 하기 |
| 3 | 계산기 에이전트 — 반복 루프로 복잡한 식 풀기 |
| 4 | 코딩 도구 — 파일 읽기/수정 함수 구현 |
| 5 | 코딩 에이전트 — LLM이 파일을 직접 수정하게 하기 |

## 0. 환경 설정

필요한 패키지를 설치하고 API 키를 불러옵니다.

In [ ]:
# 필요한 패키지 설치 (최초 1회만 실행)
# !pip install langchain langchain-openai python-dotenv

In [1]:
from dotenv import load_dotenv

# .env 파일에서 OPENAI_API_KEY 등 환경 변수를 불러옴
load_dotenv()

print("환경 변수 로드 완료")

환경 변수 로드 완료


---
## 1. LLM 호출 (LLM Call)

**핵심 개념**: `init_chat_model`로 채팅 모델을 초기화하고, `invoke()`로 질문을 보내 답을 받습니다.

```
사용자 질문 → [LLM] → 텍스트 응답
```

> **`init_chat_model`이란?**  
> LangChain의 통합 모델 초기화 함수입니다. `model_provider`만 바꾸면 OpenAI, Anthropic, Google 등 다양한 모델을 동일한 인터페이스로 사용할 수 있습니다.

In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

# LangChain으로 OpenAI 채팅 모델 초기화
# model_provider를 바꾸면 다른 회사 모델도 동일하게 사용 가능
model = init_chat_model("gpt-4.1", model_provider="openai")

def llm_call(prompt: str) -> str:
    """프롬프트를 받아 LLM 응답 텍스트를 반환하는 기본 함수"""
    # HumanMessage: 사용자의 메시지를 나타내는 LangChain 메시지 객체
    messages = [HumanMessage(content=prompt)]
    
    # invoke()로 모델 호출 → AIMessage 객체 반환
    response = model.invoke(messages)
    
    # .content로 응답 텍스트만 추출
    return response.content

In [5]:
# 테스트: 간단한 질문 보내기
prompt = "한국의 수도는 어디일까?"
response = llm_call(prompt)
print("LLM 답변:", response)

LLM 답변: 한국의 수도는 서울입니다.


---
## 2. 도구 호출 (Tool Call)

**핵심 개념**: LLM에게 사용할 수 있는 도구(함수)를 알려주면, LLM이 어떤 도구를 어떤 인자로 호출할지 결정합니다.

```
사용자 질문 → [LLM + 도구 목록] → 도구 호출 결정 → 함수 실행 → 결과
```

> **이전 방식 vs LangChain 방식**  
> - 이전: 프롬프트에 JSON 형식 지시 → LLM 응답을 직접 파싱  
> - LangChain: `@tool` 데코레이터 + `bind_tools()` → `response.tool_calls`로 자동 파싱

In [6]:
from langchain_core.tools import tool

# @tool 데코레이터: 일반 Python 함수를 LangChain 도구로 변환
# 함수 시그니처(인자 타입)와 docstring이 LLM에게 전달되는 도구 설명이 됨
@tool
def add(a: int, b: int) -> int:
    """두 수를 더합니다."""
    return a + b

# 도구 이름 → 실제 함수 매핑 (나중에 실행할 때 사용)
TOOLS_MAP = {
    "add": add,
}

# bind_tools(): 모델에게 사용 가능한 도구 목록을 알려줌
# 이제 이 모델은 필요할 때 add 도구를 호출할 수 있음
model_with_tools = model.bind_tools([add])

print("도구 바인딩 완료:", [add.name])

도구 바인딩 완료: ['add']


In [7]:
# 테스트: 덧셈이 필요한 질문 보내기
question = "1 더하기 3은 뭐야 (반드시 도구 사용)"

# 모델이 도구 호출을 결정함
response = model_with_tools.invoke([HumanMessage(content=question)])

print("LLM 응답 타입:", type(response).__name__)
print("도구 호출 목록:", response.tool_calls)

LLM 응답 타입: AIMessage
도구 호출 목록: [{'name': 'add', 'args': {'a': 1, 'b': 3}, 'id': 'call_byO1o7EwnT7TYPwWElCnRT7o', 'type': 'tool_call'}]


In [8]:
# LLM이 결정한 도구 호출을 실제로 실행하기
for tool_call in response.tool_calls:
    tool_name = tool_call["name"]    # 호출할 도구 이름
    tool_args = tool_call["args"]    # 도구에 전달할 인자 (딕셔너리)
    
    print(f"\n호출 도구: {tool_name}")
    print(f"인자: {tool_args}")
    
    # TOOLS_MAP에서 실제 함수를 찾아 실행
    if tool_name in TOOLS_MAP:
        # LangChain @tool 객체는 .invoke()로 호출 (또는 직접 함수처럼 호출 가능)
        result = TOOLS_MAP[tool_name].invoke(tool_args)
        print(f"결과: {result}")


호출 도구: add
인자: {'a': 1, 'b': 3}
결과: 4


---
## 3. 계산기 에이전트 (Calculator Agent)

**핵심 개념**: 복잡한 수식은 한 번에 풀 수 없습니다. 에이전트 루프를 통해 LLM이 단계별로 도구를 호출하며 최종 답에 도달합니다.

```
[질문] → LLM → 도구 호출 → 결과를 대화 이력에 추가
              ↑________________________↓  (도구 호출이 없을 때까지 반복)
```

> **메시지 이력 관리**  
> LangChain은 `AIMessage`(LLM 응답)와 `ToolMessage`(도구 실행 결과)를 messages 리스트에 쌓아 대화 맥락을 유지합니다.

In [9]:
from langchain_core.messages import ToolMessage

# 계산에 필요한 3가지 도구 정의
@tool
def add(a: float, b: float) -> float:
    """두 수를 더합니다."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """두 수를 곱합니다."""
    return a * b

@tool
def subtract(a: float, b: float) -> float:
    """첫 번째 숫자에서 두 번째 숫자를 뺍니다."""
    return a - b

# 계산기 도구 맵
CALC_TOOLS = [add, multiply, subtract]
CALC_TOOLS_MAP = {t.name: t for t in CALC_TOOLS}

# 3가지 도구를 모두 바인딩한 모델
calc_model = model.bind_tools(CALC_TOOLS)

print("계산기 도구 준비 완료:", list(CALC_TOOLS_MAP.keys()))

계산기 도구 준비 완료: ['add', 'multiply', 'subtract']


In [10]:
def run_calculator_agent(user_question: str, max_iterations: int = 10) -> str:
    """계산기 에이전트: 수식을 단계별 도구 호출로 풀어냄"""
    
    # 대화 이력을 리스트로 관리 (사용자 질문으로 시작)
    messages = [HumanMessage(content=user_question)]
    
    for iteration in range(max_iterations):
        print(f"\n--- 반복 {iteration + 1} ---")
        
        # LLM이 현재 메시지 이력을 보고 다음 도구를 결정
        response = calc_model.invoke(messages)
        
        # AIMessage를 이력에 추가 (LLM의 응답 저장)
        messages.append(response)
        
        # 더 이상 도구 호출이 없으면 → 최종 답 완성
        if not response.tool_calls:
            print("\n최종 답:", response.content)
            return response.content
        
        # 각 도구 호출을 실행하고 결과를 이력에 추가
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            # 실제 함수 실행
            result = CALC_TOOLS_MAP[tool_name].invoke(tool_args)
            print(f"  {tool_name}({tool_args}) → {result}")
            
            # ToolMessage: 도구 실행 결과를 대화 이력에 추가
            # tool_call_id로 어떤 도구 호출에 대한 결과인지 연결
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )
    
    return "최대 반복 횟수 초과"

In [11]:
# 테스트: (1 + 1) * 2 - 1 = 3
# 예상 흐름: add(1,1)=2 → multiply(2,2)=4 → subtract(4,1)=3
user_question = "다음 식을 계산해 주세요: (1 + 1) * 2 - 1. 반드시 각 연산마다 도구를 사용하세요."
run_calculator_agent(user_question)


--- 반복 1 ---
  add({'a': 1, 'b': 1}) → 2.0

--- 반복 2 ---
  multiply({'a': 2, 'b': 2}) → 4.0

--- 반복 3 ---
  subtract({'a': 4, 'b': 1}) → 3.0

--- 반복 4 ---

최종 답: 계산 과정은 다음과 같습니다:

1. 1 + 1 = 2
2. 2 × 2 = 4
3. 4 - 1 = 3

따라서 결과는 3입니다.


'계산 과정은 다음과 같습니다:\n\n1. 1 + 1 = 2\n2. 2 × 2 = 4\n3. 4 - 1 = 3\n\n따라서 결과는 3입니다.'

---
## 4. 코딩 도구 (Coding Tools)

**핵심 개념**: LLM 없이 순수 Python으로 파일 시스템을 조작하는 도구 3가지를 구현합니다.

| 함수 | 기능 |
|------|------|
| `list_files(directory)` | 폴더 내 파일/폴더 목록 반환 |
| `read_file(file_path)` | 파일 내용 텍스트로 반환 |
| `edit_file(file_path, old_text, new_text)` | 파일에서 문자열 교체 (없으면 새 파일 생성) |

In [12]:
import os

def list_files(directory: str = ".") -> str:
    """지정한 폴더의 파일 및 하위 폴더 목록을 반환합니다."""
    try:
        if not os.path.exists(directory):
            return f"존재하지 않는 경로입니다: {directory}"
        
        items = []
        for item in sorted(os.listdir(directory)):
            item_path = os.path.join(directory, item)
            # 폴더면 [폴더], 파일이면 [파일] 구분 표시
            if os.path.isdir(item_path):
                items.append(f"[폴더]  {item}/")
            else:
                items.append(f"[파일] {item}")
        
        if not items:
            return f"비어있는 폴더입니다: {directory}"
        
        return f"{directory}의 내용:\n" + "\n".join(items)
    except Exception as e:
        return f"오류 발생: {str(e)}"


def read_file(file_path: str) -> str:
    """파일 경로에 해당하는 텍스트 파일의 내용을 반환합니다."""
    try:
        # UTF-8 인코딩으로 파일 읽기 (한글 포함 파일 대비)
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception:
        return ""


def edit_file(file_path: str, old_text: str = "", new_text: str = "") -> str:
    """파일에서 old_text를 new_text로 교체합니다. 파일이 없으면 새로 생성합니다."""
    try:
        if os.path.exists(file_path) and old_text:
            # 기존 파일에서 텍스트 교체
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            
            if old_text not in content:
                return f"파일에서 텍스트를 찾을 수 없습니다: '{old_text}'"
            
            # 문자열 교체 후 저장
            content = content.replace(old_text, new_text)
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(content)
            return f"파일이 성공적으로 수정되었습니다: {file_path}"
        
        else:
            # 파일이 없을 경우 새로 생성 (디렉토리도 자동 생성)
            dir_name = os.path.dirname(file_path)
            if dir_name:
                os.makedirs(dir_name, exist_ok=True)
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_text)
            return f"파일이 성공적으로 생성되었습니다: {file_path}"
    except Exception as e:
        return f"파일 수정 중 오류 발생: {str(e)}"


print("코딩 도구 정의 완료: list_files, read_file, edit_file")

코딩 도구 정의 완료: list_files, read_file, edit_file


In [13]:
# 테스트 1: test_files 폴더 목록 확인
print("=== list_files 테스트 ===")
print(list_files("test_files"))

=== list_files 테스트 ===
test_files의 내용:
[파일] a.py
[파일] b.py
[파일] c.py


In [14]:
# 테스트 2: 파일 내용 읽기
print("=== read_file 테스트 ===")
print(read_file("test_files/a.py"))

=== read_file 테스트 ===
import math

def foo_renamed():
    print("hello from a")



In [15]:
# 테스트 3: 파일 수정 — b.py에서 'import math'를 'import math as m'으로 교체
print("=== edit_file 테스트 ===")
result = edit_file(
    file_path="test_files/b.py",
    old_text="import math",
    new_text="import math as m"
)
print(result)
print("\n변경 후 내용:")
print(read_file("test_files/b.py"))

=== edit_file 테스트 ===
파일이 성공적으로 수정되었습니다: test_files/b.py

변경 후 내용:
import os
import math as m as m

def bar():
    print("hello from b")



---
## 5. 코딩 에이전트 (Coding Agent)

**핵심 개념**: 섹션 3의 에이전트 루프와 섹션 4의 파일 도구를 결합합니다.  
LLM이 스스로 파일을 탐색하고, 읽고, 수정하는 **자율 코딩 에이전트**입니다.

```
[작업 지시] → LLM → list_files → read_file → edit_file → ...
                  ↑_________________________________↓  (작업 완료까지 반복)
```

> **계산기 에이전트와의 차이점**  
> 계산기 에이전트는 숫자 연산 도구를 사용했지만, 코딩 에이전트는 파일 시스템 도구를 사용합니다.  
> 에이전트 루프 구조(messages 이력 관리, tool_calls 체크)는 동일합니다.

In [16]:
# 섹션 4의 일반 함수들을 @tool 데코레이터로 감싸서 LangChain 도구로 변환

@tool
def list_files_tool(directory: str = ".") -> str:
    """지정한 폴더의 파일 및 하위 폴더 목록을 반환합니다."""
    return list_files(directory)

@tool
def read_file_tool(file_path: str) -> str:
    """파일 경로에 해당하는 텍스트 파일의 내용을 반환합니다."""
    return read_file(file_path)

@tool
def edit_file_tool(file_path: str, old_text: str = "", new_text: str = "") -> str:
    """파일에서 old_text를 new_text로 교체합니다. 파일이 없으면 새로 생성합니다."""
    return edit_file(file_path, old_text, new_text)

# 코딩 에이전트용 도구 목록
CODING_TOOLS = [list_files_tool, read_file_tool, edit_file_tool]
CODING_TOOLS_MAP = {t.name: t for t in CODING_TOOLS}

# 파일 도구를 바인딩한 코딩 에이전트 모델
coding_model = model.bind_tools(CODING_TOOLS)

print("코딩 에이전트 도구 준비 완료:", list(CODING_TOOLS_MAP.keys()))

코딩 에이전트 도구 준비 완료: ['list_files_tool', 'read_file_tool', 'edit_file_tool']


In [17]:
def run_coding_agent(user_question: str, max_iterations: int = 10) -> str:
    """코딩 에이전트: 파일 탐색/읽기/수정을 자율적으로 수행"""
    
    # 작업 지시로 대화 시작
    messages = [HumanMessage(content=user_question)]
    
    for iteration in range(max_iterations):
        print(f"\n--- 반복 {iteration + 1} ---")
        
        # LLM이 현재 상황을 파악하고 다음 도구를 결정
        response = coding_model.invoke(messages)
        
        # LLM 응답(AIMessage)을 대화 이력에 추가
        messages.append(response)
        
        # 도구 호출이 없으면 작업 완료
        if not response.tool_calls:
            print("\n작업 완료:", response.content)
            return response.content
        
        # 결정된 도구를 실행하고 결과를 이력에 추가
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            
            print(f"  [{tool_name}] 인자: {tool_args}")
            
            # 도구 실행
            result = CODING_TOOLS_MAP[tool_name].invoke(tool_args)
            print(f"  결과: {result[:100]}{'...' if len(str(result)) > 100 else ''}")
            
            # 도구 실행 결과를 ToolMessage로 이력에 추가
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )
    
    return "최대 반복 횟수 초과"

In [18]:
# 테스트: test_files 폴더의 모든 파일에서 'def foo'를 'def foo_renamed'로 교체
# 예상 흐름:
#   1. list_files_tool('test_files') → 파일 목록 확인
#   2. read_file_tool('test_files/a.py') → 내용 확인
#   3. edit_file_tool('test_files/a.py', 'def foo', 'def foo_renamed') → 수정
#   4. 나머지 파일 반복
#   5. 모든 파일 처리 완료 → 최종 답변

user_question = "test_files 폴더 내 모든 파일에서 'def foo'를 'def foo_renamed'로 바꿔줘"
run_coding_agent(user_question)


--- 반복 1 ---
  [list_files_tool] 인자: {'directory': 'test_files'}
  결과: test_files의 내용:
[파일] a.py
[파일] b.py
[파일] c.py

--- 반복 2 ---
  [edit_file_tool] 인자: {'file_path': 'test_files/a.py', 'old_text': 'def foo', 'new_text': 'def foo_renamed'}
  결과: 파일이 성공적으로 수정되었습니다: test_files/a.py
  [edit_file_tool] 인자: {'file_path': 'test_files/b.py', 'old_text': 'def foo', 'new_text': 'def foo_renamed'}
  결과: 파일에서 텍스트를 찾을 수 없습니다: 'def foo'
  [edit_file_tool] 인자: {'file_path': 'test_files/c.py', 'old_text': 'def foo', 'new_text': 'def foo_renamed'}
  결과: 파일에서 텍스트를 찾을 수 없습니다: 'def foo'

--- 반복 3 ---

작업 완료: 요청하신 작업 결과는 다음과 같습니다:

- test_files/a.py 파일에서 'def foo'가 'def foo_renamed'로 변경되었습니다.
- test_files/b.py, test_files/c.py 파일에는 'def foo'가 존재하지 않아 변경되지 않았습니다.

다른 도움이 필요하시면 말씀해 주세요!


"요청하신 작업 결과는 다음과 같습니다:\n\n- test_files/a.py 파일에서 'def foo'가 'def foo_renamed'로 변경되었습니다.\n- test_files/b.py, test_files/c.py 파일에는 'def foo'가 존재하지 않아 변경되지 않았습니다.\n\n다른 도움이 필요하시면 말씀해 주세요!"

In [19]:
# 결과 확인: 수정된 파일들 출력
print("=== 수정 결과 확인 ===")
for filename in ["a.py", "b.py", "c.py"]:
    path = f"test_files/{filename}"
    print(f"\n--- {filename} ---")
    print(read_file(path))

=== 수정 결과 확인 ===

--- a.py ---
import math

def foo_renamed_renamed():
    print("hello from a")


--- b.py ---
import os
import math as m as m

def bar():
    print("hello from b")


--- c.py ---
# No math import here
import sys

def baz():
    print("no math here")



---
## 정리

이 노트북에서 배운 내용:

1. **LLM 호출**: `init_chat_model` + `invoke([HumanMessage(...)])` 로 LLM과 대화
2. **도구 호출**: `@tool` + `bind_tools()` + `response.tool_calls` 로 LLM이 함수를 선택하게 함
3. **에이전트 루프**: `messages` 리스트에 `AIMessage` / `ToolMessage` 를 쌓아 다단계 추론 구현
4. **코딩 도구**: 파일 시스템을 조작하는 `list_files`, `read_file`, `edit_file` 구현
5. **코딩 에이전트**: 파일 도구 + 에이전트 루프 = LLM이 자율적으로 코드를 수정

### 다음 단계로 발전시키기

- `LangGraph`를 사용하면 더 복잡한 에이전트 흐름(분기, 병렬 실행) 구현 가능
- 메모리 추가: `ChatMessageHistory`로 대화 이력을 세션 간 유지
- RAG(Retrieval-Augmented Generation): 외부 문서를 검색해서 LLM에게 제공